# Experiments: Robotic Interference with Crowd Evacuation

Comparing two scenarios using the existing simulation modules:

- **Baseline** — 25 particles evacuate freely through two exits. No robot is present.
- **IK arm** — a 2-link planar arm intercepts particle flow via Jacobian-transpose IK.

All simulation logic lives in `crowd.py`, `robot.py`, `ik_arm.py`.  
This notebook only imports and calls those modules — no logic is duplicated here.

## 1. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from crowd import run_evacuation_simulation
from ik_arm import run_evacuation_with_robot_arm
from visualization import animate_evacuation

In [ ]:
# Shared scene — identical for both experiments
SEED = 42
N    = 25

exits = np.array([
    [2.0, 1.2],   # left exit
    [8.0, 1.2],   # right exit
])

walls = [
    {'a': np.array([ 0.0,  0.0]), 'b': np.array([10.0,  0.0]), 'R': 1.0, 'w': 80.0},
    {'a': np.array([10.0,  0.0]), 'b': np.array([10.0, 10.0]), 'R': 1.0, 'w': 80.0},
    {'a': np.array([10.0, 10.0]), 'b': np.array([ 0.0, 10.0]), 'R': 1.0, 'w': 80.0},
    {'a': np.array([ 0.0, 10.0]), 'b': np.array([ 0.0,  0.0]), 'R': 1.0, 'w': 80.0},
]

# Fix seed once — both experiments use the same initial positions
np.random.seed(SEED)
starts = np.column_stack([
    np.random.uniform(1.0, 9.0, N),
    np.random.uniform(3.0, 9.0, N),
])
print(f"Scene ready: {N} particles, exits at {exits.tolist()}")

## 2. Baseline Evacuation (No Robot)

Particles follow the soft-min gradient — pulled toward the closer exit,
spread across both exits by pairwise repulsion.  
The robot is **absent**; this is the uninterrupted reference case.

**Expected behaviour:** fast, smooth evacuation. Most particles reach an exit
in a few hundred steps with no detours.

In [ ]:
evac_trajs = run_evacuation_simulation(
    starts, exits, walls,
    beta=4.0, alpha=0.04, n_steps=600, tol=0.15,
    R_p=0.6, w_p=30.0,
)
print(f"Baseline: max trajectory length = {max(len(t) for t in evac_trajs)} steps")

In [ ]:
# animate_evacuation returns the FuncAnimation object — keep the reference
anim_baseline = animate_evacuation(
    evac_trajs, exits, walls,
    ee_traj=None,
    arm_angles_log=None,
    predicted_target_log=None,
    interval=30,
    title="Baseline — evacuation without robot",
)
plt.close('all')   # suppress the static inline frame; HTML cell below shows the animation

In [ ]:
HTML(anim_baseline.to_jshtml())

## 3. IK Arm Robot Interference (Phase 4)

A 2-link planar arm is rooted at `(5, 0.2)` — bottom-centre of the room.
At each timestep the robot:

1. Runs k-means (k=2) on particles near exits to find flow clusters.
2. Applies EMA smoothing (`λ=0.3`) to damp centroid jitter.
3. Predicts the cluster position `H=3` steps ahead via linear extrapolation.
4. Uses Jacobian-transpose IK to move the end-effector toward the predicted target.
5. Particles treat the end-effector as a dynamic repulsive obstacle.

**Expected behaviour:** particles near exits are deflected by the end-effector,
causing longer paths, temporary congestion, and slower evacuation.

In [ ]:
arm_base    = [5.0, 0.2]
arm_lengths = [3.0, 3.0]
arm_angles0 = [np.pi / 2, -np.pi / 6]

(arm_trajs, ee_traj, arm_angles_log,
 arm_targets, arm_cluster_log,
 arm_dominant_log, arm_smoothed_log, arm_predicted_log) = \
    run_evacuation_with_robot_arm(
        starts, exits, walls,
        beta=4.0, alpha=0.04, n_steps=600, tol=0.15,
        R_p=0.6, w_p=30.0,
        arm_base=arm_base,
        arm_angles=arm_angles0,
        arm_lengths=arm_lengths,
        alpha_ik=0.05,
        detection_radius=2.0,
        R_robot=0.9, w_robot=50.0,
        horizon=3, lambda_smooth=0.3,
    )
print(f"IK arm: max trajectory length = {max(len(t) for t in arm_trajs)} steps")

In [ ]:
anim_arm = animate_evacuation(
    arm_trajs, exits, walls,
    ee_traj=ee_traj,
    arm_angles_log=arm_angles_log,
    arm_base=arm_base,
    arm_lengths=arm_lengths,
    predicted_target_log=arm_predicted_log,
    interval=30,
    title="Phase 4 — IK arm robot interference",
)
plt.close('all')

In [ ]:
HTML(anim_arm.to_jshtml())

## 4. Comparison: Evacuation Speed

We measure how many particles **remain in the building** at each timestep.
A faster drop = faster evacuation (less interference).
A slower drop = robot successfully disrupts the flow.

In [ ]:
def remaining_over_time(trajs):
    """Number of particles still in the building at each step."""
    n_steps = max(len(t) for t in trajs)
    return np.array([sum(1 for t in trajs if len(t) > s) for s in range(n_steps)])

baseline_remaining = remaining_over_time(evac_trajs)
arm_remaining      = remaining_over_time(arm_trajs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(baseline_remaining, label='Baseline (no robot)', color='steelblue', linewidth=2)
ax.plot(arm_remaining,      label='IK arm robot',        color='darkred',   linewidth=2)
ax.set_xlabel('Timestep')
ax.set_ylabel('Particles remaining')
ax.set_title('Evacuation curve: particles remaining over time')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

base_evac = N - int(baseline_remaining[-1])
arm_evac  = N - int(arm_remaining[-1])
print(f"Baseline: {base_evac}/{N} evacuated in {len(baseline_remaining)} steps")
print(f"IK arm:   {arm_evac}/{N} evacuated in {len(arm_remaining)} steps")

### Observations

**Baseline:** particles flow directly toward both exits. The curve drops steeply
within the first few hundred steps — most particles evacuate quickly.

**IK arm:** the robot continuously repositions its end-effector to intercept
the dominant cluster approaching the nearest exit. The repulsive obstacle force causes:

- **Longer paths** — particles detour around the end-effector before reaching an exit.
- **Temporary congestion** — deflected particles join the queue for the opposite exit,
  temporarily creating a denser second cluster.
- **Slower evacuation** — the remaining-count curve falls more slowly than the baseline.

The robot does not stop evacuation entirely — particles eventually find a clear path —
but it measurably delays the flow by continuously tracking and intercepting
the densest escape stream.